# ProphetGP 사용 예시

이 노트북은 ProphetGP의 핵심 기능(학습, 후보 추천, 데이터셋 추가)을 빠르게 실행해보는 예시입니다.

In [1]:
# 필요 시 주석 해제 후 설치
# !pip install -e .[dev]

In [23]:
import numpy as np
from pathlib import Path

from prophet_gp.config import load_config
from prophet_gp.pipeline.trainer import ProphetGPPipeline
from prophet_gp.data.dataset import ReactionDatasetService

# 노트북 실행 위치가 notebooks/여도 안전하게 프로젝트 루트를 찾는다.
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "configs").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent

CONFIG_PATH = PROJECT_ROOT / "configs" / "test_20260530.yaml"
DATA_PATH = PROJECT_ROOT / "data" / "sample" / "sample_data_1.csv"
NEW_DATA_PATH = PROJECT_ROOT / "data" / "raw" / "new_batch.csv"
MERGED_OUT_PATH = PROJECT_ROOT / "data" / "raw" / "reactions_merged.csv"

config = load_config(CONFIG_PATH)
pipeline = ProphetGPPipeline(config)
dataset_service = ReactionDatasetService(config.data)

print("Project root:", PROJECT_ROOT)
print("Config loaded:", config.model_dump())

Project root: c:\Projects\ProphetGP
Config loaded: {'data': {'reactant_column': 'Mol.1', 'target_column': ['Emission Peak', 'FWHM'], 'reactant_delimiter': '|', 'ignore_columns': ['PLQY', 'Pristine Emission', 'Pristine FWHM', 'Pristine PLQY'], 'reactant_allowed_values': ['1,5-Diaminonaphthalene', '1,8-Diaminonaphthalene', '2,3-Diaminonaphthalene', '2,6-Diaminonaphthalene', '1,2-Diaminonaphthalene', '1,3-Diaminonaphthalene', '1,4-Diaminonaphthalene', '1,6-Diaminonaphthalene', '1,7-Diaminonaphthalene', '2,7-Diaminonaphthalene'], 'condition_ranges': {'Temperature': {'min': 0.0, 'max': 600.0, 'allowed_values': None, 'grid_points': 7}}, 'explicit_condition_types': {'Temperature': 'continuous'}}, 'featurization': {'featuriser': 'topo_physchem', 'combine_strategy': 'concat'}, 'optimization': {'objective': 'target', 'suggestion_strategy': 'best_output', 'standardize_gp_inputs': True, 'standardize_gp_targets': True, 'target_value': None, 'target_objectives': {'Emission Peak': {'objective': 'targ

In [24]:
# 1) 사용 가능한 featuriser 확인
available_featurisers = pipeline.featurizers.available()
print("Available featurisers count:", len(available_featurisers))
print(available_featurisers[:20])  # 앞쪽 일부만 출력

Available featurisers count: 11
['bag_of_characters', 'drfp', 'ecfp_fingerprints', 'fragments', 'molecular_graphs', 'morgan_fp', 'mqn_features', 'one_hot', 'rdkit_descriptors', 'rxnfp', 'topo_physchem']


In [25]:
# 2) 학습
artifacts = pipeline.train_from_csv(DATA_PATH)
print("Train rows:", artifacts.x_train.shape[0])
print("Feature dims:", artifacts.x_train.shape[1])
print("molecular representation check:", np.unique(artifacts.x_train[:, :-1], axis=0).shape)
print("rank of training data:", np.linalg.matrix_rank(artifacts.x_train[:, :-1]))

Train rows: 56
Feature dims: 50
molecular representation check: (41, 49)
rank of training data: 34


In [26]:
# 3) 다음 실험 조건 후보 추천 (raw + 해석 결과)
# strategy: "best_output" | "best_information"
n_candidates = 10
strategy = "best_output"
suggestions = pipeline.suggest_next_experiments(
    artifacts,
    n_candidates=n_candidates,
    strategy=strategy,
)

print("Strategy:", strategy)
print("Raw candidates shape:", suggestions.raw_candidates.shape)
print("Decoded candidates:")
for idx, row in enumerate(suggestions.decoded_candidates, 1):
    print(f"- candidate_{idx}")
    print("  predicted mean:", row["predicted_target_mean"])
    print("  predicted std:", row["predicted_target_std"])
    print("  target gap:", row["target_gap"])
    print("  nearest_known_reactants_input:", row["nearest_known_reactants_input"])
    print("  Temperature:", row["Temperature"])
    print("  mapped input:", row)

# suggestions.raw_candidates

Strategy: best_output
Raw candidates shape: (10, 50)
Decoded candidates:
- candidate_1
  predicted mean: {'Emission Peak': 496.58598790334554, 'FWHM': 82.28240807598961}
  predicted std: {'Emission Peak': 25.490494051038986, 'FWHM': 9.154269393007397}
  target gap: {'Emission Peak': 6.58598790334554, 'FWHM': None}
  nearest_known_reactants_input: 2,7-Diaminonaphthalene
  Temperature: 200.0
  mapped input: {'predicted_target_mean': {'Emission Peak': 496.58598790334554, 'FWHM': 82.28240807598961}, 'predicted_target_std': {'Emission Peak': 25.490494051038986, 'FWHM': 9.154269393007397}, 'target_gap': {'Emission Peak': 6.58598790334554, 'FWHM': None}, 'objective_score': -95.45438388268069, 'information_score': 60.13525749508537, 'total_score': -95.45438388268069, 'ranking_strategy': 'best_output', 'mapped_reactants_input': '2,7-Diaminonaphthalene', 'mapped_reactants_smiles': ['Nc1ccc2ccc(N)cc2c1'], 'nearest_known_reactants_input': '2,7-Diaminonaphthalene', 'nearest_known_reactants_smiles':

In [27]:
# 3b) 지정 입력에 대한 예측 (GP posterior mean / std)
# suggest_next_experiments와 달리, 사용자가 정한 반응물·조건에 대한 예측값을 조회한다.
query_inputs = [
    {"reactants": "2,6-Diaminonaphthalene", "Temperature": 300.0},
    {"reactants": "5-amino-1,10-phenanthroline|salicylic acid", "Temperature": 250.0},
]
prediction_result = pipeline.predict_targets(artifacts, query_inputs)

for idx, row in enumerate(prediction_result.predictions, 1):
    print(f"- query_{idx}")
    print("  reactants:", row["reactants_input"])
    print("  conditions:", row["conditions"])
    print("  predicted mean:", row["predicted_target_mean"])
    print("  predicted std:", row["predicted_target_std"])
    print("  target gap:", row["target_gap"])

- query_1
  reactants: 2,6-Diaminonaphthalene
  conditions: {'Temperature': 300.0}
  predicted mean: {'Emission Peak': 521.0493244388341, 'FWHM': 86.56962892029868}
  predicted std: {'Emission Peak': 43.91566511265839, 'FWHM': 11.701095643453689}
  target gap: {'Emission Peak': 31.049324438834105, 'FWHM': None}
- query_2
  reactants: 5-amino-1,10-phenanthroline|salicylic acid
  conditions: {'Temperature': 250.0}
  predicted mean: {'Emission Peak': 581.4490889516505, 'FWHM': 87.59222873208027}
  predicted std: {'Emission Peak': 41.94735539595289, 'FWHM': 12.338842524122727}
  target gap: {'Emission Peak': 91.44908895165054, 'FWHM': None}


In [ ]:
# 4) 신규 배치 데이터 append
# 파일이 준비되어 있지 않으면 이 셀은 건너뛰세요.
merged = dataset_service.append_csv(DATA_PATH, NEW_DATA_PATH, MERGED_OUT_PATH)
print("Merged rows:", len(merged))
print("Saved to:", MERGED_OUT_PATH)